<a href="https://colab.research.google.com/github/srisahana33-sys/University_RAG/blob/main/bahubali4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf transformers accelerate

In [ ]:
import os
from typing import List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from sentence_transformers import CrossEncoder
from transformers import pipeline

In [ ]:
# add ur file path
PDF_PATH = ""
INDEX_PATH = "faiss_index"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 250

EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

TOP_K_RETRIEVE = 20
TOP_K_RERANK = 3

In [ ]:
from google import genai
#add ur gemini api key
client = genai.Client(
    api_key=""
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello, world!"
)

In [ ]:
# chunking
def load_and_chunk(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    chunks = splitter.split_documents(documents)

    print(f"Pages: {len(documents)} | Chunks: {len(chunks)}")
    return chunks

In [ ]:
#vector db
def get_vector_db(chunks=None):
    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

    if os.path.exists(INDEX_PATH):
        print("Loading existing FAISS index...")
        db = FAISS.load_local(
            INDEX_PATH,
            embeddings,
            allow_dangerous_deserialization=True
        )
    else:
        print("Building FAISS index...")
        db = FAISS.from_documents(chunks, embeddings)
        db.save_local(INDEX_PATH)

    return db


def get_retriever(db):
    return db.as_retriever(search_kwargs={"k": TOP_K_RETRIEVE})

reranker = CrossEncoder(RERANK_MODEL)

def rerank(query, docs, top_k=TOP_K_RERANK):
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_k]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
prompt = PromptTemplate.from_template("""
Answer the question ONLY from the provided context. If the answer is not found in the context, state "The provided document does not contain information to answer this question."

Question:
{question}

Context:

{context}

Answer:
""")

In [ ]:
def llm(prompt_text):
    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt_text)

    return response.text.replace(prompt_text, '').strip()

class RAGSystem:
    def __init__(self, retriever):
        self.retriever = retriever

    def query(self, question: str) -> str:
        # Step 1: Retrieve
        docs = self.retriever.invoke(question)

        # Step 2: Rerank
        top_docs = rerank(question, docs)

        # Step 3: Build context
        context = "\n\n".join([d.page_content for d in top_docs])

        # Step 4: Prompt
        final_prompt = prompt.format(context=context, question=question)

        # Step 5: Generate
        answer = llm(final_prompt)

        return answer

chunks = load_and_chunk(PDF_PATH)
db = get_vector_db(chunks)
retriever = get_retriever(db)

rag = RAGSystem(retriever)

Pages: 53 | Chunks: 483


/tmp/ipykernel_1835/2972844417.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS index...


In [ ]:
print(rag.query("What does course plan consists of?"))


A course plan consists of a list of lectures/ experiments carried out in each instructional class/ lab by the course teacher during the semester as per the LTPC of the course, with details like mode of delivery, reference material used, and others.


In [ ]:
print(rag.query("how to make mutton biryani"))


The provided document does not contain information to answer this question.


In [ ]:
print(rag.query("Can a student take unlimited courses in a summer semester? and is there any way a freshman can take the summer sem new course?"))


Regarding the first question, the provided document states that there is no minimum or maximum number of credits fixed for course registration during summer semesters. If courses are offered and there is no clash in the timetable, a student is permitted to register for any number of courses. However, the document also specifies that "The maximum number of courses to be taken, eligibility criteria to register and related information shall be specified through Circulars issued by the University from time to time." Therefore, while this document does not set a fixed limit, the number of courses is not unlimited as a maximum will be specified via university circulars.

Regarding the second question, the document mentions that weekend intra-semester and summer semesters are conducted to help students clear their backlogs. It also states that eligibility criteria to register for these semesters shall be specified through Circulars issued by the University. The provided document does not cont

In [ ]:
print(rag.query("What happens if a student is debarred from writing FAT?what to do now so that i can write my exams .."))


If a student is debarred from writing FAT, they shall be awarded an "N" grade for that course, and they have to re-register the course again and clear it with a performance grade.

The provided document does not contain information on what to do now to write the exams immediately after being debarred. The context indicates that the student must re-register the course again.


In [ ]:
print(rag.query("whether alcohol is allowed??"))

Possession or consumption of alcohol is not allowed within the Institute or during travel authorized by the Institute.


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def respond(message, history):
    return rag.query(message)

chatbot_ui = gr.ChatInterface(
    fn=respond,
    title="chatbot",
    description=""
)
chatbot_ui.launch(share=True)

In [ ]:
!git init
!git add .
!git commit -m "First commit"
!git branch -M main
!git remote add origin https://github.com//repo-name.git
!git push -u origin main